# Задание 13: модели E5-small, E5-large и LaBSE

In [2]:
import re
import math
import numpy as np
from collections import Counter, defaultdict
import os

In [4]:
from razdel import sentenize

with open("corpus.txt", "r", encoding="utf-8") as f:
    corpus_text = f.read().strip()

corpus_texts = [s.text for s in sentenize(corpus_text)]

In [5]:
CORPUS_SIZE = len(corpus_texts)

Заведем матрицу релевантностей rel_q и оценим каждое предложения для всех запросов

In [6]:
rel_q = 3 * [[0] * CORPUS_SIZE]

In [7]:
rel_q[0][0] = 2
rel_q[0][1] = 1
rel_q[0][2] = 2
rel_q[0][3] = 1

In [8]:
rel_q[1][73] = 1
rel_q[1][74] = 1
rel_q[1][75] = 2

In [9]:
rel_q[2][185] = 2
rel_q[2][186] = 2
rel_q[2][187] = 1
rel_q[2][188] = 1
rel_q[2][189] = 1
rel_q[2][191] = 1
rel_q[2][192] = 1
rel_q[2][196] = 1
rel_q[2][198] = 1
rel_q[2][200] = 1
rel_q[2][201] = 1
rel_q[2][204] = 1
rel_q[2][206] = 1
rel_q[2][211] = 1
rel_q[2][213] = 1
rel_q[2][225] = 1
rel_q[2][226] = 1
rel_q[2][229] = 1
rel_q[2][236] = 1
rel_q[2][237] = 1
rel_q[2][221] = 1
rel_q[2][230] = 1
rel_q[2][231] = 1
rel_q[2][232] = 2
rel_q[2][233] = 2
rel_q[2][300] = 1

In [10]:
import numpy as np
rel_q_sorted = np.sort(rel_q, axis=1)[:, ::-1]

In [11]:
from sentence_transformers import SentenceTransformer

models = {
    "e5-small": SentenceTransformer("intfloat/multilingual-e5-small"),
    "e5-large": SentenceTransformer("intfloat/multilingual-e5-large"),
    "labse": SentenceTransformer("sentence-transformers/LaBSE")
}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

In [12]:
import numpy as np
from tqdm import tqdm

corpus_embeddings = {}

In [14]:
corpus_embeddings['e5-small'] = models['e5-small'].encode(
        corpus_texts,
        convert_to_numpy=True,
        batch_size=32,
        show_progress_bar=True
    )

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

In [15]:
corpus_embeddings['e5-large'] = models['e5-large'].encode(
        corpus_texts,
        convert_to_numpy=True,
        batch_size=32,
        show_progress_bar=True
    )

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

In [16]:
corpus_embeddings['labse'] = models['labse'].encode(
        corpus_texts,
        convert_to_numpy=True,
        batch_size=32,
        show_progress_bar=True
    )

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

In [17]:
def cosine(a, b):
    an = np.linalg.norm(a)
    bn = np.linalg.norm(b)
    if an == 0 or bn == 0:
        return 0
    return float(np.dot(a, b) / (an * bn))


In [18]:
def search(model_name, query, q_num, topk=10):
    model = models[model_name]
    q_vec = model.encode([query], convert_to_numpy=True)[0]
    emb = corpus_embeddings[model_name]
    # косинусные близости
    sims = np.array([cosine(q_vec, e) for e in emb])
    top_idx = sims.argsort()[::-1][:topk]
    relevances = [rel_q[q_num][i] for i in top_idx]
    top_sims = sims[top_idx]

    return relevances, top_idx, top_sims


In [19]:
import math

def ndcg_k(relevances, rel_ideal, k=10):
    rel_ideal = rel_ideal[:k]
    discounts = []
    for position in range(1, k + 1):
        discount = math.log2(position + 1)
        discounts.append(discount)

    # DCG
    dcg = 0
    for i in range(k):
        gain = relevances[i] / discounts[i]
        dcg += gain

    # IDCG
    idcg = 0
    for i in range(k):
        gain = rel_ideal[i] / discounts[i]
        idcg += gain

    return dcg / idcg if idcg > 0 else 0


In [20]:
test_queries = [
    "Памятник на месте рождения Пушкина стоит не там.",
    "Одна из самых известных сцен в истории Голливуда происходит на кукурузном поле.",
    "Победный гол сальвадорского футболиста (на илл.) в ворота соперника привёл к шестидневной войне."
]

In [23]:
topk = 10
results = {name: [] for name in models.keys()}

for q_num, query in enumerate(test_queries):
    print("\n\n======================================================")
    print("Запрос:", query)
    for name in models.keys():
        rels, idxs, sims = search(name, query, q_num, topk)
        results[name].append(rels)
        print(f"\n--- {name} ---")
        for rank, (r, i, s) in enumerate(zip(rels, idxs, sims), start=1):
            print(f"{rank}. ({s:.4f}) {corpus_texts[i]}")




Запрос: Памятник на месте рождения Пушкина стоит не там.

--- e5-small ---
1. (0.8712) Памятник Пушкину на Бауманской улице в Москве — скульптурное изображение (бюст), установленное на месте предполагаемого рождения русского поэта и писателя Александра Сергеевича Пушкина.
2. (0.8674) Более поздние исследования, проведённые уже после установки памятника, показывают, что место рождения Пушкина скорее находится на соседней Малой Почтовой улице.
3. (0.8666) Отличие памятника от многочисленных других памятников Пушкину состоит в том, что он изображает поэта в юном возрасте.
4. (0.8585) == История ==
Памятник был сооружён вблизи того места, где ранее находился дом по Немецкой (ныне — Бауманская) улице, в котором, как предполагалось, 26 мая (6 июня) 1799 года родился русский поэт и писатель Александр Сергеевич Пушкин (1799—1837).
5. (0.8377) На стене соседней школы находится мемориальная доска из красного гранита с бронзовым барельефом Пушкина, созданная скульптором Константином Кошкиным.
6

In [24]:
print("NDCG@10:\n")

for name in models.keys():
    print(f"Модель {name}:")
    for i in range(3):
        score = ndcg_k(results[name][i], rel_q_sorted[i])
        print(f"Запрос {i+1}: {score:.4f}")
    print()


NDCG@10:

Модель e5-small:
Запрос 1: 0.4881
Запрос 2: 0.2445
Запрос 3: 0.2981

Модель e5-large:
Запрос 1: 0.3698
Запрос 2: 0.2445
Запрос 3: 0.3531

Модель labse:
Запрос 1: 0.5033
Запрос 2: 0.4794
Запрос 3: 0.3865



**Выводы:**   
Модель E5-small для запроса q1 показала себя лучше, чем E5-large, это может быть связано с тем, что для данного запроса количество релевантных предложений было небольшим, и маленькая модель лучше справилась в нахождении нужных предложений, не выявляя сложных зависимостей.

Модель E5-large лучше справислась с запросом q3, где число подходящих текстов больше, и модель показывала результат 0.3531 по сравнению с 0.2981 для модели E5-small. Отсюда можно понять, что большие модели стоит использовать в случае большого размера корпуса с подходящими кандидатами.

Модель LaBSE показала самые лучшие результаты. Для запроса q2 обе версии E5 показывают одинаково низкие значения NDCG, тогда как LaBSE почти вдвое превосходит их. Скорее всего модель LaBSE лучше улавливает связи в русскоязычных текстах и не теряется, когда в тексте встречаются слова на других языках. Для второго запроса в корпусе присутствуют предложения на английском языке с названиями фильмов, модель LaBSE сумела не растеряться в этих нерелевантных предложениях. Преобладание по качеству для всех запросов говорит о качестве обучающих данных для модели LaBSE.
